MD# 2. Groundwater level models, trends and drought indicators
Pastas recharge model (rain − f·evaporation, Gamma response) with a linear trend, Theil–Sen / Mann–Kendall trends on monthly levels, the Standardised Groundwater Index (SGI) and monthly percentile drought thresholds. The full run for all stations is `scripts/pastas_trend_drought.py`; this notebook walks through one station.

In [ ]:
import os, sys, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else ".")
DATA = os.path.join(ROOT, "data"); sys.path.insert(0, os.path.join(ROOT, "scripts"))
import pastas as ps
from scipy.stats import theilslopes, kendalltau
ps.set_log_level('ERROR')

In [ ]:
STATION = "Rwonyo"
st = pd.read_csv(os.path.join(DATA, "raw", "Stations.csv")).set_index("station_name")
d = pd.read_csv(os.path.join(DATA, "raw", "levels", st.loc[STATION, "data_file"]), encoding="latin1", on_bad_lines="skip")
dc = d.columns[0]; d[dc] = pd.to_datetime(d[dc], errors="coerce"); d = d.dropna(subset=[dc]).set_index(dc).sort_index(); d = d[~d.index.duplicated()]
dep = pd.to_numeric(d.iloc[:, 0], errors="coerce").dropna(); dep = dep[(dep - dep.median()).abs() < 5 * dep.std()]
dep.plot(figsize=(10, 3), title=f"{STATION}: depth to water (m bgl)").invert_yaxis()

MD## Forcing: ERA5-Land rain and evaporation at the station

In [ ]:
tp = pd.read_csv(os.path.join(DATA, "processed", "era5_tp_stations.csv"), index_col=0, parse_dates=True)[STATION].dropna()
pev = pd.read_csv(os.path.join(DATA, "processed", "era5_pev_stations.csv"), index_col=0, parse_dates=True)[STATION].dropna()
h = -dep.loc[max(dep.index[0], tp.index[0]):min(dep.index[-1], tp.index[-1])]
ml = ps.Model(h, name=STATION)
ml.add_stressmodel(ps.RechargeModel(tp / 1000, pev / 1000, ps.Gamma(), name="recharge", recharge=ps.rch.Linear()))
ml.add_stressmodel(ps.LinearTrend(h.index[0], h.index[-1], name="trend"))
ml.solve(report=False)
print(f"EVP {ml.stats.evp():.1f}%, RMSE {ml.stats.rmse():.2f} m")
ml.plots.results(figsize=(10, 7));

MD## Observed trend (Theil–Sen, Mann–Kendall)

In [ ]:
mon = dep.resample("MS").mean().dropna()
yrs = (mon.index - mon.index[0]).days / 365.25
slope = -theilslopes(mon.values, yrs)[0]; p = kendalltau(yrs, mon.values).pvalue
print(f"trend {slope:+.3f} m/yr (positive = rising), Mann-Kendall p = {p:.3g}")

MD## Drought: SGI and monthly thresholds

In [ ]:
sgi = ps.stats.sgi(-mon)
ax = sgi.plot(kind="line", figsize=(10, 2.8)); ax.axhline(-1, ls="--", c="orange"); ax.axhline(-1.5, ls="--", c="r"); ax.set_title("SGI")
thr = mon.groupby(mon.index.month).quantile([0.5, 0.8, 0.9]).unstack().round(2)
thr.columns = ["normal_P50", "moderate_P80", "severe_P90"]; thr

MD## All stations
Results written by `pastas_trend_drought.py`:

In [ ]:
s = pd.read_csv(os.path.join(DATA, "processed", "pastas_trend_drought_summary.csv"))
s[["station", "EVP", "t50_days", "obs_trend_m_per_yr", "trend_p_value", "last_SGI", "status"]].round(2).sort_values("EVP", ascending=False)